# 02 — Profit and loss

[Module 02](../course/02-trading-terminology.md) · [Notebook index](README.md)

**Purpose:** Estimate gross and net results from executed prices. All numbers are illustrative. Run cells in order.

For a linear Forex contract, with direction $d=+1$ for buy and $d=-1$ for sell:

$$\text{Gross P/L}=d\times(\text{Exit}-\text{Entry})\times\text{Lots}\times\text{Contract size}\times\text{Conversion rate}$$

Conversion rate = account-currency units per unit of quote currency. Enter actual or assumed execution prices (ask/bid as appropriate); the spread is already reflected in those prices. Never subtract spread twice.

$$\text{Net P/L}=\text{Gross P/L}-\text{Commission}-\text{Other fees}+\text{Signed swap}$$

Commission and other fees are positive total charges in account currency; swap is negative for a charge and positive for a credit. Realised P/L refers to closed trades; unrealised P/L changes as open positions are repriced.


In [ ]:
from math import isfinite, isclose

def positive(value, name):
    if not isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be finite and positive")
    return value

def finite(value, name):
    if not isfinite(value):
        raise ValueError(f"{name} must be finite")
    return value

def expect_value_error(function, *args):
    try:
        function(*args)
    except ValueError:
        return
    raise AssertionError("Expected ValueError")

def forex_gross_pnl(side, entry, exit_price, lots, contract_size, quote_to_account=1.0):
    if side not in ("buy", "sell"):
        raise ValueError("side must be buy or sell")
    positive(entry, "entry")
    positive(exit_price, "exit price")
    direction = 1 if side == "buy" else -1
    return (direction * (exit_price - entry) * positive(lots, "lots")
            * positive(contract_size, "contract size")
            * positive(quote_to_account, "conversion rate"))

def net_pnl(gross, commission=0.0, swap=0.0, other_fees=0.0):
    finite(gross, "gross")
    finite(swap, "swap")
    for value, name in ((commission, "commission"), (other_fees, "other fees")):
        if not isfinite(value) or value < 0:
            raise ValueError(f"{name} must be finite and nonnegative")
    return gross - commission - other_fees + swap


## Worked example

Buy EUR/USD at an executed 1.1002, close at an executed 1.1050, volume 0.10 lots and contract size 100,000. Gross P/L is $48. With $1.20 total commission and a $0.30 swap charge, net P/L is $46.50. The spread is already captured in execution prices.


In [ ]:
gross = forex_gross_pnl("buy", 1.1002, 1.1050, 0.10, 100_000)
net = net_pnl(gross, commission=1.20, swap=-0.30)
print(f"Gross: USD {gross:.2f}")
print(f"Net: USD {net:.2f}")
print(f"Illustrative gross in ZAR: {forex_gross_pnl('buy', 1.1002, 1.1050, 0.10, 100_000, 18):.2f}")


## Try it yourself

Sell EUR/USD at 1.1000 and close at 1.0960, 0.10 lots. Assume $1 round-trip commission and no swap. Expected net: $39. Then reverse the price direction and check the loss. 


In [ ]:
assert isclose(forex_gross_pnl("buy", 1.1002, 1.1050, 0.10, 100_000), 48)
assert isclose(net_pnl(48, 1.20, -0.30), 46.50)
assert isclose(net_pnl(forex_gross_pnl("sell", 1.1000, 1.0960, 0.10, 100_000), 1), 39)
assert isclose(forex_gross_pnl("sell", 1.1000, 1.1040, 0.10, 100_000), -40)
assert isclose(forex_gross_pnl("buy", 1.1002, 1.1000, 0.10, 100_000), -2)
assert isclose(net_pnl(10, 1, 2, 0.5), 10.5)
expect_value_error(forex_gross_pnl, "hold", 1.1, 1.2, 0.1, 100_000)
expect_value_error(forex_gross_pnl, "buy", 1.1, 1.2, 0, 100_000)
expect_value_error(net_pnl, 10, -1)
print("All P/L checks passed.")


## Online calculator and limitations

- [Myfxbook profit calculator](https://www.myfxbook.com/forex-calculators/profit-calculator)
- [Myfxbook swap calculator](https://www.myfxbook.com/forex-calculators/swap-calculator)
- [MQL5 OrderCalcProfit](https://www.mql5.com/en/docs/trading/ordercalcprofit)

Compare identical entry/exit prices, volume, conversion and costs. Some tools show gross results only. Conversion rates and charges can vary. This formula is for linear FX contracts, not all instrument calculation modes; use broker/platform estimates for other products. It does not predict future slippage, commissions or swaps.
